# 2b — Audit the frozen Stage 1 variants and manifest

Run this after notebook 2 and before any policy outcomes. Pulling notebook updates changes the benchmark Git SHA, so this notebook refreshes only manifest provenance from the already frozen `stage1_resolved_variants.csv` when necessary. It never reruns variant selection.

In [ ]:
import csv
import subprocess
from collections import Counter
from pathlib import Path

home = Path.home()
repo = home / "async-vla-latency-bench"
stage0 = home / "stage0"
stage1 = home / "stage1"
plus = home / "LIBERO-plus"
python = home / "venv-stage1-ood/bin/python"
variants_path = stage1 / "stage1_resolved_variants.csv"
manifest_path = stage1 / "stage1_manifest.csv"

for required in (repo, stage0 / "selected_high_delay.json", variants_path, manifest_path, python):
    if not required.exists():
        raise SystemExit(f"STOP: missing prerequisite {required}")

current_sha = subprocess.run(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
plus_sha = subprocess.run(
    ["git", "-C", str(plus), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
manifest_before = list(csv.DictReader(open(manifest_path)))
manifest_shas = {row['git_sha'] for row in manifest_before}
print("repository SHA:", current_sha)
print("manifest SHA(s) before audit:", manifest_shas)


In [ ]:
# Refresh identity only; the frozen variant CSV is the immutable selection input.
if manifest_shas != {current_sha}:
    subprocess.run([
        str(python), "-m", "async_vla_benchmark.scripts.make_stage1_manifest",
        "--variants", str(variants_path),
        "--selected-delay", str(stage0 / "selected_high_delay.json"),
        "--output", str(manifest_path),
        "--git-sha", current_sha,
        "--lerobot-git-sha", "2aba372b4e217cc47db28e0f836859b20d1456c9",
        "--libero-plus-git-sha", plus_sha,
        "--model-revision", "8e174154ef5f6c60a8da12ae99c303d8963138c1",
    ], cwd=repo, check=True)
    subprocess.run([
        str(python), "-m", "async_vla_benchmark.scripts.import_stage0_controls",
        "--manifest", str(manifest_path),
        "--stage0-dir", str(stage0),
        "--stage1-dir", str(stage1),
    ], cwd=repo, check=True)
    print("Refreshed manifest provenance without changing frozen variants")
else:
    print("Manifest provenance already matches repository")


In [ ]:
variants = list(csv.DictReader(open(variants_path)))
manifest = list(csv.DictReader(open(manifest_path)))

assert len(variants) == 21
assert len({(row['task_key'], row['perturbation_key']) for row in variants}) == 21
assert all(int(row['api_task_index']) == int(row['classification_id']) - 1 for row in variants)
assert all(row['difficulty_level'] not in ('', 'None', 'null') for row in variants)
assert Counter(row['task_key'] for row in variants) == {
    'spatial_transport': 7, 'goal_drawer': 7, 'long_stove_moka': 7,
}
assert all(count == 3 for count in Counter(row['perturbation_key'] for row in variants).values())

print("Selected difficulties:", Counter(row['difficulty_level'] for row in variants))
print("Non-L2 selections:")
for row in variants:
    if row['difficulty_level'] != '2':
        print(row)

assert len(manifest) == 480
assert len({row['run_id'] for row in manifest}) == 480
assert {row['git_sha'] for row in manifest} == {current_sha}
assert Counter(row['scene_condition'] for row in manifest) == {'ood': 420, 'id': 60}
assert Counter(row['seed'] for row in manifest) == {str(seed): 96 for seed in range(5)}
assert Counter(row['execution_method'] for row in manifest) == {'naive_async': 240, 'rtc': 240}
assert Counter(row['delay_condition'] for row in manifest) == {'low': 240, 'high': 240}
assert sum(row['reuse_stage0'].lower() == 'true' for row in manifest) == 24
assert all(int(row['added_delay_ms']) == (0 if row['delay_condition'] == 'low' else 200) for row in manifest)
assert all(row['n_action_steps'] == '25' for row in manifest)

print("PASS: frozen Stage 1 manifest audit")
print("manifest=480 OOD=420 ID=60 reused=24 new=456")
print("STOP HERE and paste this output before notebook 3.")
